In [ ]:
# Cell 1: Install Dependencies with NumPy fix
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install "numpy<2.0"  # Fix NumPy version first
    !pip install unsloth
else:
    !pip install "numpy<2.0"  # Fix NumPy version first
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3
    !pip install --no-deps unsloth

# Cell 2: Load Model (your current approach is correct)
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Patch the config to include ignore_index
model.config.ignore_index = -100

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

# Cell 3-4: Data Loading (your current approach is correct)
import pandas as pd
from datasets import Dataset

df = pd.read_csv("final_dataset_combined.csv")
print(f"Loaded {len(df)} rows with columns: {df.columns.tolist()}")

def convert_to_gemma3_format(df):
    formatted_data = []
    for idx, row in df.iterrows():
        user_content = row['instruction']
        if pd.notna(row['input']) and str(row['input']).strip():
            user_content += f"\n\n{row['input']}"
        
        conversation = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": row['output']}
        ]
        formatted_data.append({"conversations": conversation})
    return formatted_data

converted_data = convert_to_gemma3_format(df)
dataset = Dataset.from_list(converted_data)

# Cell 5: Format and Tokenize the 'conversations' field
def tokenize_and_format(examples):
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    outputs = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=2048,
        return_tensors="pt"
    )
    return {
        "input_ids": outputs["input_ids"],
        "attention_mask": outputs["attention_mask"]
    }

dataset = dataset.map(
    tokenize_and_format,
    batched=True,
    remove_columns=["conversations"]  # this is the correct column to remove
)

print(f"Dataset columns after tokenization: {dataset.column_names}")

# Cell 5.1: Save Formatted Gemma-3 Dataset for Later Use
import os
import json
from datasets import Dataset

print("=== SAVING FORMATTED GEMMA-3 DATASET ===")

# Save the dataset in multiple formats for flexibility
save_directory = "gemma3_formatted_dataset"
os.makedirs(save_directory, exist_ok=True)

# 1. Save as HuggingFace Dataset (recommended)
dataset.save_to_disk(os.path.join(save_directory, "hf_dataset"))
print(f"✓ Saved HuggingFace dataset to: {save_directory}/hf_dataset")

# 2. Save as JSON for maximum compatibility
dataset_json = []
for i in range(len(dataset)):
    dataset_json.append({
        "text": dataset[i]["text"],
        "index": i
    })

with open(os.path.join(save_directory, "gemma3_formatted.json"), "w", encoding="utf-8") as f:
    json.dump(dataset_json, f, ensure_ascii=False, indent=2)
print(f"✓ Saved JSON dataset to: {save_directory}/gemma3_formatted.json")

# 3. Save as CSV for easy inspection
import pandas as pd
df_formatted = pd.DataFrame({"text": [dataset[i]["text"] for i in range(len(dataset))]})
df_formatted.to_csv(os.path.join(save_directory, "gemma3_formatted.csv"), index=False, encoding="utf-8")
print(f"✓ Saved CSV dataset to: {save_directory}/gemma3_formatted.csv")

# 4. Save metadata about the dataset
metadata = {
    "original_dataset_size": len(df),
    "formatted_dataset_size": len(dataset),
    "columns": dataset.column_names,
    "sample_text_length": len(dataset[0]["text"]),
    "format": "Gemma-3 chat template",
    "template_parts": {
        "user_start": "<start_of_turn>user",
        "assistant_start": "<start_of_turn>model",
        "turn_end": "<end_of_turn>"
    },
    "created_from": "final_dataset_combined.csv",
    "total_examples": len(dataset)
}

with open(os.path.join(save_directory, "dataset_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Saved metadata to: {save_directory}/dataset_metadata.json")

print(f"\n=== DATASET SAVED SUCCESSFULLY ===")
print(f"Location: {os.path.abspath(save_directory)}")
print(f"Total examples: {len(dataset)}")
print(f"Sample text preview:")
print("="*50)
print(dataset[0]["text"][:300] + "..." if len(dataset[0]["text"]) > 300 else dataset[0]["text"])
print("="*50)

# Show file sizes
for file_name in os.listdir(save_directory):
    file_path = os.path.join(save_directory, file_name)
    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"📁 {file_name}: {size_mb:.2f} MB")

# Cell 6: Training with All Fixes
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "input_ids",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        
        # All the critical fixes:
        remove_unused_columns = False,
        dataloader_pin_memory = False,
        dataloader_drop_last = False,
        group_by_length = False,
        bf16 = False,
        fp16 = False,
    ),
)


# Apply response masking
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

# Cell 7: Train
try:
    trainer_stats = trainer.train()
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"[ERROR] Training failed: {e}")

# Cell 8: Save
model.save_pretrained("gemma-3-kiran-finetune")
tokenizer.save_pretrained("gemma-3-kiran-finetune")

In [ ]:
# Cell 1: Install Dependencies (from working_gemma3.ipynb)
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.1.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3
    !pip install --no-deps unsloth




# Cell 2: Load Model (following working_gemma3.ipynb pattern)
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    # token = "hf_...", # use one if using gated models
)

# Add LoRA adapters
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

# Set up Gemma-3 chat template
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)


# Cell 3: Load and Analyze Your Dataset
import pandas as pd
from datasets import Dataset

# Load your CSV dataset
print("Loading your dataset...")
df = pd.read_csv("final_dataset_combined.csv")
print(f"Loaded {len(df)} rows with columns: {df.columns.tolist()}")

# Check the format of your data
print("\nFirst few rows of your dataset:")
print(df.head())

# Sample check
sample_row = df.iloc[0]
print(f"\nSample data structure:")
print(f"Instruction: {sample_row['instruction'][:100]}...")
print(f"Input: {sample_row['input'][:100] if pd.notna(sample_row['input']) else 'No input'}...")
print(f"Output: {sample_row['output'][:100]}...")


# Cell 4: Convert Your Dataset to Gemma-3 Format
def convert_to_gemma3_format(df):
    """Convert instruction/input/output format to Gemma-3 conversations format"""
    formatted_data = []
    
    for idx, row in df.iterrows():
        # Create user message by combining instruction and input
        user_content = row['instruction']
        if pd.notna(row['input']) and str(row['input']).strip():
            user_content += f"\n\n{row['input']}"
        
        # Create conversation in the format Gemma-3 expects
        conversation = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": row['output']}
        ]
        
        formatted_data.append({"conversations": conversation})
    
    return formatted_data

# Convert your data
print("Converting data to Gemma-3 conversations format...")
converted_data = convert_to_gemma3_format(df)
dataset = Dataset.from_list(converted_data)

print(f"Successfully converted {len(dataset)} examples")
print("\nSample conversation structure:")
print(dataset[0]['conversations'])



# Cell 5: Apply Gemma-3 Chat Template (CORRECTED VERSION)
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, 
            tokenize=False, 
            add_generation_prompt=False
        ).removeprefix('<bos>')  # Remove <bos> as in official_gemma.ipynb
        for convo in convos
    ]
    return {"text": texts}

# Apply the formatting and REMOVE the conversations column
print("Applying Gemma-3 chat template...")
dataset = dataset.map(
    formatting_prompts_func, 
    batched=True,
    remove_columns=["conversations"]  # ADD THIS LINE - removes the problematic column
)

print("✓ Dataset formatted for Gemma-3")
print(f"Dataset columns: {dataset.column_names}")

# Show how Gemma-3 formats the conversation
print("\nSample formatted text (Gemma-3 format):")
print("="*50)
print(dataset[0]["text"])
print("="*50)

print(f"\nDataset ready with {len(dataset)} examples for training!")



# Cell 6: Training Setup (CORRECTED VERSION with dtype fixes)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,  # CHANGED: Reduced from 2 to 1
        gradient_accumulation_steps = 8,  # CHANGED: Increased from 4 to 8
        warmup_steps = 20,  # CHANGED: Increased from 5 to 20
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 10,  # CHANGED: Increased from 1 to 10
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        
        # CRITICAL ADDITIONS for dtype consistency:
        bf16 = False,  # NEW: Explicitly disable bf16
        fp16 = False,  # NEW: Explicitly disable fp16
        dataloader_pin_memory = False,  # NEW: Better for V100
        remove_unused_columns = False,  # NEW: Prevent column issues
        dataloader_drop_last = False,  # NEW: Additional safety
        group_by_length = False,  # NEW: Disable grouping
        # Remove dataset_num_proc = 2,  # REMOVED: Can cause issues
    ),
)

# Apply response-only training (following official_gemma.ipynb)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
    # REMOVED: num_proc argument (not needed with datasets==3.1.0)
)

print("✓ Trainer configured for Gemma-3 fine-tuning")
print("✓ Response-only training enabled")


# Cell 6.5: Dtype Consistency Fix (ADD THIS CELL)
import torch

print("Checking and fixing dtype consistency...")

# Check current dtypes
print("Current model dtypes:")
for name, param in model.named_parameters():
    if 'lm_head' in name or 'embed' in name:
        print(f"{name}: {param.dtype}")

# Force lm_head to consistent dtype
if hasattr(model, 'base_model'):
    if hasattr(model.base_model, 'lm_head'):
        print("Fixing lm_head dtype...")
        model.base_model.lm_head = model.base_model.lm_head.to(torch.float32)
        print(f"lm_head dtype now: {model.base_model.lm_head.weight.dtype}")
    
    # Also check language_model if it exists
    if hasattr(model.base_model, 'language_model') and hasattr(model.base_model.language_model, 'lm_head'):
        print("Fixing language_model.lm_head dtype...")
        model.base_model.language_model.lm_head = model.base_model.language_model.lm_head.to(torch.float32)
        print(f"language_model.lm_head dtype now: {model.base_model.language_model.lm_head.weight.dtype}")

print("✓ Dtype consistency check completed")

# Cell 7: Memory Check and Training
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# Start training
trainer_stats = trainer.train()

# Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

# Cell 8: Save Model
model.save_pretrained("gemma-3-kiran-finetune")
tokenizer.save_pretrained("gemma-3-kiran-finetune")
print("Model saved successfully!")

# Cell 9: Test the Model
messages = [{
    "role": "user",
    "content": "Who are you?"
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 128,
    temperature = 1.0, 
    top_p = 0.95, 
    top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)



